# weight-decay-l2-add — worked example 1: Fold L2 decay into the gradient

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `weight-decay-l2-add`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

L2 (classic) weight decay folds `lmda * theta` into the gradient before the optimizer's momentum/variance update: `g = g + lmda * theta`. A guard `if lmda != 0` skips the fold (and its allocation) when decay is disabled. The fold returns a new tensor rather than mutating `g` in place.

## Worked solution

We augment a raw gradient with the L2 weight-decay term.

1. We check `if lmda != 0` — when decay is off we return `g` unchanged, the same object, avoiding a needless allocation in the hot loop.
2. Otherwise we compute `g + lmda * theta`. This adds a force proportional to the parameter that pulls weights toward zero.
3. We return a fresh tensor (`+` does not mutate `g`), so the caller's original gradient is preserved.

We print the augmented gradient for a positive `theta` and confirm the zero-lambda path returns the identical object.

In [ ]:
import torch as t

def apply_weight_decay(theta, g, lmda):
    if lmda != 0:
        g = g + lmda * theta
    return g

theta = t.tensor([1.0, -2.0, 3.0])
g = t.tensor([0.5, 0.5, 0.5])
out = apply_weight_decay(theta, g, lmda=0.1)
print('augmented:', out.tolist())
bypass = apply_weight_decay(theta, g, lmda=0.0)
print('bypass is same object:', bypass is g)